# 10 distinct and dropDuplicates

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Remove duplicate rows with <code>distinct</code> and <code>dropDuplicates</code>, deduplicate on a subset of columns, and keep a <b>specific</b> row per key (such as the latest record), which is the pattern used in real pipelines.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Duplicates are everywhere: retried API calls, files loaded twice, CDC streams that replay events. Deduplication is a core Silver-layer step and a standard interview question: "How do you keep the latest record per customer?" Both <code>distinct</code> and <code>dropDuplicates</code> are <b>wide</b> transformations, so they cost a shuffle.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The three tools</b>
<ul><li><code>distinct()</code> removes rows that are identical in <b>every</b> column</li><li><code>dropDuplicates()</code> with no arguments does the same as <code>distinct()</code></li><li><code>dropDuplicates(["col1", "col2"])</code> keeps <b>one</b> row per combination of those columns, and keeps <b>all</b> columns. Which row survives is <b>not guaranteed</b></li><li>To control which row is kept (latest, highest, first), use a <b>window function</b> with <code>row_number()</code></li><li><code>dropDuplicatesWithinWatermark</code> is the streaming version (lesson in folder 08)</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A CRM sends customer updates every hour. The same customer can appear 5 times in one day as their address changes. The Silver <code>customers</code> table must keep exactly one row per <code>customer_id</code>: the one with the latest <code>updated_at</code>. <code>dropDuplicates(["customer_id"])</code> would keep a random version, which is a subtle bug. <code>row_number()</code> ordered by <code>updated_at desc</code> keeps the right one.
</div>

## Syntax

```python
df.distinct()                                  # exact duplicate rows
df.dropDuplicates()                            # same as distinct()
df.dropDuplicates(["customer_id"])             # one row per key, arbitrary row kept

# Keep the LATEST row per key (deterministic)
from pyspark.sql import Window
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
latest = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates customer update events with an exact duplicate row and several versions per customer.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
An 8-row table. Rows 1 and 2 are exact duplicates. <code>C1</code> has 3 versions, <code>C2</code> has 2, and <code>C3</code> has 2 with the same timestamp.
</div>

In [0]:
from pyspark.sql import functions as F

updates = spark.createDataFrame(
    [
        ("C1", "Amal", "Chennai", "2024-06-01 09:00:00"),
        ("C1", "Amal", "Chennai", "2024-06-01 09:00:00"),   # exact duplicate (replayed message)
        ("C1", "Amal", "Bangalore", "2024-06-03 12:00:00"),
        ("C2", "John", "London", "2024-06-02 08:00:00"),
        ("C2", "John", "Manchester", "2024-06-05 17:30:00"),
        ("C3", "Vivek", "Dubai", "2024-06-04 10:00:00"),
        ("C3", "Vivek", "Abu Dhabi", "2024-06-04 10:00:00"),  # same timestamp: a tie
        ("C4", "Priya", "Mumbai", "2024-06-01 07:45:00"),
    ],
    "customer_id STRING, name STRING, city STRING, updated_at STRING",
).withColumn("updated_at", F.to_timestamp("updated_at"))
display(updates)

## 1. distinct: remove exact duplicates

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Removes rows that match in every column, and compares the counts.<br><br>
<b>Why it matters</b><br>It is the first, cheapest check for replayed messages or files loaded twice.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
8 rows before, 7 after. Only the exact copy of the first row is removed.
</div>

In [0]:
print("Before:", updates.count(), "| After distinct:", updates.distinct().count())

# distinct on selected columns: unique values of one or more columns
updates.select("customer_id").distinct().orderBy("customer_id").show()

## 2. dropDuplicates on a subset of columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Keeps one row per <code>customer_id</code>, then one row per <code>customer_id</code> and <code>city</code>.<br><br>
<b>Why it matters</b><br>Unlike <code>select(...).distinct()</code>, <code>dropDuplicates(subset)</code> keeps <b>all</b> columns. But it gives no control over which duplicate survives.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Non-deterministic</b><br>
Which row survives <code>dropDuplicates(["customer_id"])</code> depends on partitioning and execution order. It can change between runs and between clusters. Never rely on it for "latest record" logic.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows (one per customer, with whichever city Spark happened to keep), then 7 unique customer and city pairs.
</div>

In [0]:
updates.dropDuplicates(["customer_id"]).orderBy("customer_id").show()
print("Unique (customer, city) pairs:", updates.dropDuplicates(["customer_id", "city"]).count())

## 3. Keep the latest record per key (the production pattern)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Ranks each customer's rows by <code>updated_at</code> descending with <code>row_number()</code>, and keeps rank 1.<br><br>
<b>Why it matters</b><br>This is the deterministic way to deduplicate, and the most important pattern in this lesson. Window functions are covered fully in lesson 19.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows: C1 in Bangalore, C2 in Manchester, C4 in Mumbai, and C3 with <b>either</b> Dubai or Abu Dhabi, because the timestamps tie (see section 4).
</div>

In [0]:
from pyspark.sql import Window

w_latest = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

latest = (
    updates
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
latest.orderBy("customer_id").show()

## 4. Edge case: ties need a tie-breaker

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a second ordering column so that ties are resolved the same way every time.<br><br>
<b>Why it matters</b><br>If two rows share the same <code>updated_at</code>, <code>row_number</code> picks one arbitrarily. Add a deterministic tie-breaker: an ingestion sequence number, a source file name, or (as a last resort) a value column.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C3 is always <code>Abu Dhabi</code>, because <code>city</code> ascending is the tie-breaker and "Abu Dhabi" sorts before "Dubai". The <code>max_by</code> version gives the same latest city for C1, C2 and C4. For the tied C3 its choice is not guaranteed.
</div>

In [0]:
w_tiebreak = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc(), F.col("city").asc())

latest_det = (
    updates.withColumn("rn", F.row_number().over(w_tiebreak)).filter("rn = 1").drop("rn")
)
latest_det.orderBy("customer_id").show()

# Alternative for "value at the latest time": max_by(value, ordering column) inside groupBy.
updates.groupBy("customer_id").agg(
    F.max_by("city", "updated_at").alias("latest_city"),
    F.max("updated_at").alias("latest_update"),
).orderBy("customer_id").show()

## 5. Finding duplicates before removing them

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts how many rows each key has, and lists the keys that appear more than once.<br><br>
<b>Why it matters</b><br>In a data quality check you often want to <b>report</b> duplicates (and maybe quarantine them) rather than silently remove them.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C1 (3 rows), C2 (2) and C3 (2) are reported as duplicate keys. The duplicate ratio is printed as a percentage.
</div>

In [0]:
dupes = (
    updates.groupBy("customer_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .orderBy(F.col("row_count").desc())
)
dupes.show()

total = updates.count()
unique_keys = updates.select("customer_id").distinct().count()
print(f"Duplicate ratio: {(total - unique_keys) / total:.0%}")

## 6. Edge case: nulls and case differences

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows that <code>distinct</code> treats nulls as equal to each other, and that <code>"amal"</code> and <code>"Amal"</code> are different values.<br><br>
<b>Why it matters</b><br>Deduplication often needs normalization first (trim, lower case), otherwise near-duplicates survive.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 distinct raw rows (the two null rows collapse into one), then 2 after normalizing names with <code>lower(trim(...))</code>.
</div>

In [0]:
messy = spark.createDataFrame([("Amal",), ("amal ",), (None,), (None,)], "name STRING")
print("Raw distinct:", messy.distinct().count())
print("Normalized distinct:", messy.select(F.lower(F.trim("name")).alias("name")).distinct().count())

## Under the hood

```
df.dropDuplicates(["customer_id"])
        |
        v
 Logical plan:  Deduplicate [customer_id]
        |
        v
 Optimizer:     rewritten to an Aggregate: group by customer_id, first(other columns)
        |
        v
 Physical plan: HashAggregate (partial)  -> Exchange hashpartitioning(customer_id) -> HashAggregate (final)
                                             ^^^^^^^^ shuffle
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `distinct` and `dropDuplicates` must bring equal keys together on one executor, so they **shuffle** data.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> lazy, like every transformation.

Spark first deduplicates **within** each partition (partial aggregate) to reduce what it shuffles, then deduplicates again after the shuffle. The `first(...)` in the plan is why the kept row is arbitrary. Look for `Exchange` and `first` in the plan below.

In [0]:
(
    spark.range(1_000_000)
    .withColumn("customer_id", F.col("id") % 1000)
    .dropDuplicates(["customer_id"])
).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: different rows kept on every run</b><br>
<code>dropDuplicates(subset)</code> is non-deterministic. Use <code>row_number()</code> over a window with an explicit order and a tie-breaker.<br><br>
**⛔ Problem: duplicates remain after <code>distinct()</code>**<br>Rows differ in some column, often a timestamp, whitespace or letter case. Normalize, or deduplicate on the business key only.<br><br>
<b>⛔ Problem: lost columns after deduplication</b><br><code>select("a").distinct()</code> keeps only <code>a</code>. Use <code>dropDuplicates(["a"])</code> to keep all columns.<br><br>
<b>⛔ Problem: dedup is slow on big data</b><br>It shuffles. Filter and select only the needed columns first, and deduplicate once, not repeatedly.<br><br>
<b>⛔ Problem: incremental loads still create duplicates in the target</b><br>Deduplicating the batch isn't enough when the target already has the key. Use <code>MERGE</code> (Delta lesson 07).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>distinct()</code> and <code>dropDuplicates()</code>?</b><br>
With no arguments they are the same. <code>dropDuplicates</code> also accepts a subset of columns and keeps one row per combination of those columns while returning all columns.<br><br>

<b>🎤 2. Which row does <code>dropDuplicates(["id"])</code> keep?</b><br>
An arbitrary one. It is compiled into an aggregate with <code>first()</code>, so the result depends on partitioning and task order and isn't guaranteed.<br><br>

<b>🎤 3. How do you keep the latest record per key?</b><br>
A window partitioned by the key, ordered by the timestamp descending plus a tie-breaker, then <code>row_number()</code> and filter where it equals 1. <code>max_by</code> in a groupBy is an alternative when I only need a few columns.<br><br>

<b>🎤 4. Are <code>distinct</code> and <code>dropDuplicates</code> narrow or wide?</b><br>
Wide. Equal keys must meet on the same executor, so they cause a shuffle, with a partial aggregation before it to reduce data.<br><br>

<b>🎤 5. How does <code>distinct</code> treat nulls?</b><br>
Nulls are treated as equal to each other for deduplication, so several null rows collapse into one.<br><br>

<b>🎤 6. How do you deduplicate in a streaming pipeline?</b><br>
With <code>dropDuplicatesWithinWatermark</code> or <code>dropDuplicates</code> combined with a watermark, so Spark can drop old state. For upserts into a table, <code>MERGE</code> is used. Both come later in the streaming folder.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A table has duplicate <code>order_id</code> rows with different <code>updated_at</code> values. Which approach reliably keeps the most recent row per order?</b><br>
A. <code>df.dropDuplicates(["order_id"])</code><br>
B. <code>df.distinct()</code><br>
C. <code>row_number()</code> over a window partitioned by <code>order_id</code> ordered by <code>updated_at DESC</code>, then filter rank 1<br>
D. <code>df.orderBy("updated_at").dropDuplicates(["order_id"])</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> A and D don't guarantee which row survives (an <code>orderBy</code> before <code>dropDuplicates</code> isn't preserved through the shuffle). B removes only exact duplicates.</details><br><br>

<b>Q2. Which SQL statement returns the unique combinations of <code>country</code> and <code>city</code>?</b><br>
A. <code>SELECT UNIQUE country, city FROM t</code><br>
B. <code>SELECT DISTINCT country, city FROM t</code><br>
C. <code>SELECT country, DISTINCT city FROM t</code><br>
D. <code>SELECT country, city FROM t GROUP BY ALL DISTINCT</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>DISTINCT</code> applies to the whole selected row. <code>GROUP BY country, city</code> would also work.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>updates</code>:
<ol><li>Count exact duplicate rows (total rows minus distinct rows)</li><li>Return the unique list of cities in alphabetical order</li><li>Keep one row per <code>name</code> using <code>dropDuplicates</code>, and explain in a comment why the city may vary</li><li>Keep the <b>earliest</b> record per customer, with <code>city</code> ascending as a tie-breaker</li><li>Report customers who changed city at least once (more than one distinct city)</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
from pyspark.sql import Window

# 1. Exact duplicates
print("Exact duplicate rows:", updates.count() - updates.distinct().count())

# 2. Unique cities
updates.select("city").distinct().orderBy("city").show()

# 3. One row per name: the kept row is arbitrary because dropDuplicates uses first() after a shuffle
updates.dropDuplicates(["name"]).show()

# 4. Earliest per customer, deterministic
w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").asc(), F.col("city").asc())
updates.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn").orderBy("customer_id").show()

# 5. Customers with more than one distinct city
updates.groupBy("customer_id").agg(F.countDistinct("city").alias("cities")).filter("cities > 1").show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>distinct()</code> = <code>dropDuplicates()</code> = remove rows identical in every column</li><li><code>dropDuplicates(subset)</code> keeps one row per key with all columns, but <b>which</b> row is arbitrary</li><li>For "latest per key" use <code>row_number()</code> over a window ordered by time desc, plus a tie-breaker</li><li><code>max_by(col, order_col)</code> returns a value from the latest row inside <code>groupBy</code></li><li>Both are <b>wide</b> transformations: partial dedup, shuffle, final dedup</li><li>Normalize (trim, lower case) before deduplicating text</li><li>Report duplicates with <code>groupBy(key).count().filter("count &gt; 1")</code></li></ul>
</div>

| Task | Code |
|---|---|
| Exact duplicates | `df.distinct()` |
| Per key, any row | `df.dropDuplicates(["k"])` |
| Latest per key | `row_number().over(Window.partitionBy("k").orderBy(desc("ts")))` then `rn = 1` |
| Latest value | `F.max_by("v", "ts")` |
| Find duplicate keys | `df.groupBy("k").count().filter("count > 1")` |
| SQL | `SELECT DISTINCT a, b FROM t` |

## Git commit

```
git add 01_spark_basics/10_distinct_dropduplicates.ipynb
git commit -m "add 10 distinct dropduplicates notebook"
```